# Job Application Sidekick (LangGraph)

I built this as a practical copilot for my DevOps/Cloud job search.

What it helps me do:

- find relevant roles online
- score each role against my profile (0-100) and explain why it is a fit
- compare those roles against my profile
- draft and refine human-sounding cover letters
- apply guardrails so I do not make fake claims or sound robotic
- keep sensitive info safer by using a local sanitized `candidate_profile.txt`

What it does **not** do:

- it does not auto-apply, auto-login, or auto-submit applications on my behalf

I still review everything and submit manually. So this project is mainly a copilot for research + writing, with human approval as part of the workflow.

In [ ]:
from typing import TypedDict, List, Dict, Any, Optional

import gradio as gr
import re
import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

In [ ]:
load_dotenv(override=True)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

In [ ]:
class JobMatch(BaseModel):
    title: str
    company: str
    location: str
    url: str
    match_score: int = Field(ge=0, le=100)
    why_fit: str


class JobMatchList(BaseModel):
    matches: List[JobMatch]


class ApplicationDraft(BaseModel):
    role_title: str
    company: str
    tailored_summary: str
    cover_letter: str
    guardrail_notes: List[str]


class GuardrailReview(BaseModel):
    approved: bool
    issues: List[str]
    revised_cover_letter: str


class SidekickState(TypedDict):
    query: str
    cv_text: str
    jobs_raw: List[Dict[str, Any]]
    matched_jobs: List[Dict[str, Any]]
    selected_job_index: int
    application_draft: Dict[str, Any]
    approval_required: bool


def redact_pii(text: str) -> str:
    """Basic PII scrubber for emails, phones, links, and location lines."""
    redacted = text
    redacted = re.sub(r"[\w\.-]+@[\w\.-]+", "[REDACTED_EMAIL]", redacted)
    redacted = re.sub(r"\+?\d[\d\s\-\(\)]{7,}\d", "[REDACTED_PHONE]", redacted)
    redacted = re.sub(r"https?://\S+|www\.\S+|linkedin\.com/\S+|github\.com/\S+", "[REDACTED_LINK]", redacted)
    redacted = re.sub(r"(?im)^.*(chester, united kingdom|united kingdom|uk).*$", "[REDACTED_LOCATION]", redacted)
    return redacted.strip()

In [ ]:
def _fallback_jobs() -> List[Dict[str, Any]]:
    """Fallback sample jobs so the workflow remains demonstrable if APIs are unavailable."""
    return [
        {
            "title": "Senior DevOps Engineer",
            "company": "ExampleCloud",
            "location": "Remote",
            "url": "https://example.com/jobs/senior-devops-engineer",
            "description": "AWS, Terraform, Kubernetes, CI/CD, observability, incident response.",
        },
        {
            "title": "Cloud Platform Engineer",
            "company": "InfraScale",
            "location": "Remote - Europe",
            "url": "https://example.com/jobs/cloud-platform-engineer",
            "description": "GCP/AWS platform, IaC, GitOps, SRE, cost optimization.",
        },
        {
            "title": "Site Reliability Engineer",
            "company": "OpsReliant",
            "location": "Remote",
            "url": "https://example.com/jobs/sre",
            "description": "SLI/SLO, Prometheus/Grafana, Kubernetes reliability engineering.",
        },
    ]


def _normalize_remotive(jobs: List[Dict[str, Any]], limit: int) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for j in jobs[:limit]:
        out.append(
            {
                "title": j.get("title", ""),
                "company": j.get("company_name", ""),
                "location": j.get("candidate_required_location", "Remote"),
                "url": j.get("url", ""),
                "description": j.get("description", ""),
            }
        )
    return out


def _normalize_arbeitnow(jobs: List[Dict[str, Any]], limit: int) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for j in jobs[:limit]:
        out.append(
            {
                "title": j.get("title", ""),
                "company": j.get("company_name", ""),
                "location": j.get("location", "Remote"),
                "url": j.get("url", ""),
                "description": j.get("description", ""),
            }
        )
    return out


def _filter_for_devops_cloud(jobs: List[Dict[str, Any]], query: str, limit: int) -> List[Dict[str, Any]]:
    tokens = [
        t.strip().lower()
        for t in re.split(r"\s+|\||,|or", query)
        if t.strip() and t.strip().lower() not in {"and", "the"}
    ]
    if not tokens:
        return jobs[:limit]

    filtered = []
    for j in jobs:
        text = f"{j.get('title','')} {j.get('description','')}".lower()
        if any(tok in text for tok in tokens):
            filtered.append(j)
    return (filtered or jobs)[:limit]


def search_remote_jobs(query: str, limit: int = 12) -> List[Dict[str, Any]]:
    """Fetch jobs from multiple sources, then filter locally for better reliability."""
    combined: List[Dict[str, Any]] = []
    errors: List[str] = []

    # Source 1: Remotive broad pull (no strict search param)
    try:
        r = requests.get("https://remotive.com/api/remote-jobs", timeout=25)
        r.raise_for_status()
        payload = r.json()
        combined.extend(_normalize_remotive(payload.get("jobs", []), limit=50))
    except Exception as exc:
        errors.append(f"remotive: {exc}")

    # Source 2: Arbeitnow public board
    try:
        r2 = requests.get("https://www.arbeitnow.com/api/job-board-api", timeout=25)
        r2.raise_for_status()
        payload2 = r2.json()
        combined.extend(_normalize_arbeitnow(payload2.get("data", []), limit=50))
    except Exception as exc:
        errors.append(f"arbeitnow: {exc}")

    # De-duplicate by URL
    seen = set()
    deduped = []
    for j in combined:
        u = j.get("url", "")
        if u and u not in seen:
            deduped.append(j)
            seen.add(u)

    filtered = _filter_for_devops_cloud(deduped, query=query, limit=limit)

    if not filtered:
        print(f"Live APIs returned 0 filtered jobs. Using fallback. Errors: {' | '.join(errors) if errors else 'none'}")
        return _fallback_jobs()

    print(f"Fetched {len(filtered)} jobs from live APIs")
    return filtered


def discover_jobs(state: SidekickState) -> Dict[str, Any]:
    jobs = search_remote_jobs(state["query"])
    return {"jobs_raw": jobs}

In [ ]:
def match_jobs_to_cv(state: SidekickState) -> Dict[str, Any]:
    if not state["jobs_raw"]:
        return {"matched_jobs": []}

    short_jobs = [
        {
            "title": j["title"],
            "company": j["company"],
            "location": j["location"],
            "url": j["url"],
            "description": j["description"][:1200],
        }
        for j in state["jobs_raw"][:10]
    ]

    prompt = f"""
You are a career assistant.
Given the CV and job list, return the best 3 matches for DevOps/Cloud roles.
Score each match from 0-100, and explain briefly why the candidate fits.

CV:
{state['cv_text'][:5000]}



Jobs:
{short_jobs}
"""

    structured = llm.with_structured_output(JobMatchList)
    result = structured.invoke(prompt)
    return {"matched_jobs": [m.model_dump() for m in result.matches]}

In [ ]:
def draft_application(state: SidekickState) -> Dict[str, Any]:
    matches = state["matched_jobs"]
    if not matches:
        return {
            "application_draft": {
                "role_title": "",
                "company": "",
                "tailored_summary": "No matching jobs were found.",
                "cover_letter": "",
                "guardrail_notes": ["No jobs found to draft for."],
            },
            "approval_required": False,
        }

    idx = min(state.get("selected_job_index", 0), len(matches) - 1)
    target = matches[idx]

    prompt = f"""
Write a tailored application draft for this candidate.

Candidate CV:
{state['cv_text'][:5000]}

Target role:
{target}

Rules:
- Be natural and human, not robotic.
- Do not invent any experience, certification, or impact not grounded in the CV.
- Keep cover letter between 140 and 220 words.
- Tone should be confident, specific, and respectful.
"""

    structured = llm.with_structured_output(ApplicationDraft)
    result = structured.invoke(prompt)
    return {
        "application_draft": result.model_dump(),
        "approval_required": True,
    }

In [ ]:
def apply_cover_letter_guardrails(state: SidekickState) -> Dict[str, Any]:
    draft = state["application_draft"]
    if not draft or not draft.get("cover_letter"):
        return {}

    review_prompt = f"""
Review this cover letter.

CV:
{state['cv_text'][:5000]}

Draft:
{draft['cover_letter']}

Guardrails:
1) No fake claims or unverifiable achievements.
2) Avoid robotic/AI-sounding phrases.
3) Keep it concise, natural, and specific to the role.

Return approved=true only if all guardrails pass. If not approved, rewrite it.
"""

    reviewer = llm.with_structured_output(GuardrailReview)
    review = reviewer.invoke(review_prompt)

    if review.approved:
        notes = list(draft.get("guardrail_notes", [])) + ["Guardrails passed."]
        draft["guardrail_notes"] = notes
    else:
        draft["cover_letter"] = review.revised_cover_letter
        notes = list(draft.get("guardrail_notes", [])) + [
            f"Guardrails triggered: {', '.join(review.issues) if review.issues else 'rewritten for quality.'}"
        ]
        draft["guardrail_notes"] = notes

    return {"application_draft": draft}

In [ ]:
builder = StateGraph(SidekickState)

builder.add_node("discover_jobs", discover_jobs)
builder.add_node("match_jobs_to_cv", match_jobs_to_cv)
builder.add_node("draft_application", draft_application)
builder.add_node("apply_cover_letter_guardrails", apply_cover_letter_guardrails)

builder.add_edge(START, "discover_jobs")
builder.add_edge("discover_jobs", "match_jobs_to_cv")
builder.add_edge("match_jobs_to_cv", "draft_application")
builder.add_edge("draft_application", "apply_cover_letter_guardrails")
builder.add_edge("apply_cover_letter_guardrails", END)

sidekick_graph = builder.compile()

## Run the sidekick

The notebook loads `candidate_profile.txt` and redacts PII before processing.

The result gives you:
- discovered jobs count (API or fallback)
- top matched jobs
- one tailored cover letter draft for the selected job
- guardrail notes
- approval flag (human-in-the-loop)

In [ ]:
# Recommended: load sanitized candidate profile from local file
with open("candidate_profile.txt", "r", encoding="utf-8") as f:
    cv_text = f.read()

# Extra safety: apply local PII redaction before any LLM calls
cv_text = redact_pii(cv_text)

In [ ]:
if not cv_text.strip():
    raise ValueError("candidate_profile.txt appears empty. Add your sanitized profile first.")

initial_state: SidekickState = {
    "query": "devops engineer OR cloud engineer OR sre OR platform engineer",
    "cv_text": cv_text,
    "jobs_raw": [],
    "matched_jobs": [],
    "selected_job_index": 0,
    "application_draft": {},
    "approval_required": True,
}

result = sidekick_graph.invoke(initial_state)

print(f"Raw jobs discovered: {len(result['jobs_raw'])}")
print("Top Matches:")
for i, m in enumerate(result["matched_jobs"]):
    print(f"{i+1}. {m['title']} @ {m['company']} | score={m['match_score']} | {m['url']}")

print("\n--- Draft ---")
print(result["application_draft"].get("cover_letter", ""))
print("\nGuardrail Notes:", result["application_draft"].get("guardrail_notes", []))
print("Approval Required:", result["approval_required"])

In [ ]:
def run_sidekick(query: str, cv_text: str, selected_job_index: int = 0) -> Dict[str, Any]:
    state: SidekickState = {
        "query": query,
        "cv_text": cv_text,
        "jobs_raw": [],
        "matched_jobs": [],
        "selected_job_index": selected_job_index,
        "application_draft": {},
        "approval_required": True,
    }
    return sidekick_graph.invoke(state)


def revise_cover_letter(cv_text: str, job: Dict[str, Any], current_cover_letter: str, feedback: str) -> str:
    prompt = f"""
Rewrite this cover letter based on user feedback while following guardrails.

CV:
{cv_text[:5000]}

Job:
{job}

Current cover letter:
{current_cover_letter}

User feedback:
{feedback}

Guardrails:
- Keep it human and natural.
- Do not invent experience or achievements.
- Keep between 140 and 220 words.
- Make it specific to role and company.
"""
    review = llm.with_structured_output(GuardrailReview).invoke(prompt)
    return review.revised_cover_letter if review.revised_cover_letter else current_cover_letter

## Optional mini UI (refine + approve)

Use this if you want a quick interactive loop for tweaking your cover letter before final use.

In [ ]:
def ui_generate(query: str, cv_text: str, selected_job_index: int):
    result = run_sidekick(query, cv_text, selected_job_index)
    matches = result.get("matched_jobs", [])
    draft = result.get("application_draft", {})

    if matches:
        lines = [
            f"{i+1}. {m['title']} @ {m['company']} (score={m['match_score']})\n{m['url']}"
            for i, m in enumerate(matches)
        ]
        matches_text = "\n\n".join(lines)
        target = matches[min(selected_job_index, len(matches)-1)]
    else:
        matches_text = "No matches found."
        target = {}

    return (
        matches_text,
        draft.get("cover_letter", ""),
        "\n".join(draft.get("guardrail_notes", [])),
        target,
    )


def ui_revise(cv_text: str, target_job: Dict[str, Any], cover_letter: str, feedback: str):
    if not cover_letter.strip() or not feedback.strip():
        return cover_letter
    return revise_cover_letter(cv_text, target_job, cover_letter, feedback)


with gr.Blocks() as demo:
    gr.Markdown("## Job Application Sidekick")

    query = gr.Textbox(
        label="Job search query",
        value="devops engineer cloud engineer kubernetes terraform aws",
    )
    cv = gr.Textbox(label="CV text", lines=12)
    idx = gr.Number(label="Select job index (0-based)", value=0, precision=0)

    generate_btn = gr.Button("Generate Matches + Draft")
    matches_box = gr.Textbox(label="Top Matches", lines=10)
    cover_box = gr.Textbox(label="Cover Letter Draft", lines=12)
    notes_box = gr.Textbox(label="Guardrail Notes", lines=4)
    target_state = gr.State({})

    feedback = gr.Textbox(label="Revision feedback", placeholder="e.g. Make this more concise and mention Kubernetes migration project")
    revise_btn = gr.Button("Revise Draft")

    generate_btn.click(
        ui_generate,
        inputs=[query, cv, idx],
        outputs=[matches_box, cover_box, notes_box, target_state],
    )
    revise_btn.click(
        ui_revise,
        inputs=[cv, target_state, cover_box, feedback],
        outputs=[cover_box],
    )

# Launch locally:
demo.launch()

## Next step (manual apply)

Once you approve the draft:
1. Open the chosen job URL
2. Paste approved cover letter
3. Submit manually

This keeps applications compliant with platform terms and avoids accidental submissions.